In [12]:
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import Whitespace

reviews = [
    "the movie was great",
    "i hated this film",
    "what a wonderful story and beautiful acting",
    "boring plot and terrible dialogue",
    "i loved every minute of it",
    "the worst movie i have ever seen",
    "a masterpiece from start to finish",
    "waste of time and money",
    "the acting was brilliant but the ending was weak",
    "i would watch this again and again",
    "so slow and predictable",
    "amazing visuals and a moving soundtrack",
    "the characters felt flat and lifeless",
    "one of the best films of the year",
    "i fell asleep halfway through",
    "funny heartfelt and very well made",
    "the story made no sense at all",
    "great direction and a strong cast",
    "disappointing sequel to a good original",
    "i enjoyed this much more than i expected",
]

tokenizer = Tokenizer(BPE(unk_token="UNK"))
tokenizer.pre_tokenizer = Whitespace()
trainer = BpeTrainer(vocab_size=200, special_tokens=["UNK"])
tokenizer.train_from_iterator(reviews, trainer)

In [50]:
tokenizer.encode("the movie was great").tokens


['the', 'movie', 'was', 'gr', 'eat']

In [ ]:
# By hand:
from collections import Counter
corpus = "lo low lowest slow slower newest widest new here"

vocab = {tuple(list(w)+["_"]): c for w , c in Counter(corpus.split()).items()}
vocab

In [ ]:
for step in range(6):
    pairs = Counter()
    for word, freq in vocab.items():
        for a, b in zip(word[:-1], word[1:]): pairs[(a, b)] += freq
    best = pairs.most_common(1)[0][0]     # the most frequent pair
    vocab = merge(vocab, best)               # glue it everywhere

# Transformers

In [15]:
from transformers import pipeline
clf = pipeline("sentiment-analysis")          # downloads a trained model once
clf("movie was not so good")

[transformers] No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f.
Using a pipeline without specifying a model name and revision in production is not recommended.


config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

c:\Users\Hp\.conda\envs\rnn-gpu\lib\site-packages\huggingface_hub\file_download.py:141: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Hp\.cache\huggingface\hub\models--distilbert--distilbert-base-uncased-finetuned-sst-2-english. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

[{'label': 'NEGATIVE', 'score': 0.9997442364692688}]

# Byte pair encoding

In [18]:
import torch
import torch.nn as nn
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.pre_tokenizers import Whitespace
from tokenizers.trainers import BpeTrainer

In [48]:
reviews = [
    "the movie was great",
    "i hated this film",
    "what a wonderful story and beautiful acting",
    "boring plot and terrible dialogue",
    "i loved every minute of it",
    "the worst movie i have ever seen",
    "a masterpiece from start to finish",
    "waste of time and money",
    "the acting was brilliant but the ending was weak",
]
labels = [1,0,1,0,1,0,1,0,1]

assert len(reviews) == len(labels), "reviews and labels must be the same length"

#Bpe is the algorithm that decides how to chop text into pieces
#Tokenizer is a wrapper tha hold algorithm plusgives you .encode() , .decode()
tokenizer = Tokenizer(BPE(unk_token="[UNK]"))
#runs before bpe , splits on space , punctuations
tokenizer.pre_tokenizer = Whitespace()
#The trainer is the recipe , not the tokenizer itself stop once the vocabulary hits 100 entries
#reserve two slots at the very front for PAD and UNK

trainer = BpeTrainer(vocab_size=100,special_tokens=["[PAD]" , "[UNK]"])
#Think of it as glueing letters together m greedily
tokenizer.train_from_iterator(reviews , trainer)

raw_ids = [tokenizer.encode(r).ids for r in reviews]
print("Raw Ids:" , raw_ids)

print(tokenizer.encode("the movie was great").tokens)
print(tokenizer.encode("the movie was great").ids)



Raw Ids: [[31, 53, 29, 73, 69], [10, 45, 42, 28, 48, 72, 13], [97, 2, 98, 99, 38, 37, 23, 33, 63, 39, 78, 54], [61, 30, 88, 19, 33, 50, 89, 6, 65, 35, 74, 6], [10, 35, 94, 55, 23, 81, 39, 6, 49, 77], [31, 95, 38, 53, 10, 75, 6, 55, 90, 14], [2, 82, 50, 87, 64, 70, 85, 38, 57, 19, 92, 71, 48, 9], [29, 91, 49, 93, 6, 33, 36, 84], [31, 54, 29, 60, 47, 80, 83, 62, 31, 68, 30, 29, 96, 11]]
['the', 'movie', 'was', 'gr', 'eat']
[31, 53, 29, 73, 69]


In [ ]:
#Getting the actual vocab size: as if bpe stops early can overshoot 
# so recheck the vocabulary size hich will be later used for embedding
VOCAB_SIZE = tokenizer.get_vocab_size()
print("Actual vocab size : "  , VOCAB_SIZE)
raw_ids = [tokenizer.encode(r).ids for r in reviews]
print("raw ids : " , raw_ids)

# Pad and truncate to a fixed length 

max_len = max(len(ids) for ids in raw_ids)
print("max_len choosen :" , max_len) #14

padded_ids = []
for ids in raw_ids:
    ids = ids[:max_len] # cut if too long : Keep the first 14 
    ids = ids + [0] * (max_len - len(ids))
    padded_ids.append(ids)
print("Padded ids:", padded_ids)

Actual vocab size :  100
raw ids :  [[31, 53, 29, 73, 69], [10, 45, 42, 28, 48, 72, 13], [97, 2, 98, 99, 38, 37, 23, 33, 63, 39, 78, 54], [61, 30, 88, 19, 33, 50, 89, 6, 65, 35, 74, 6], [10, 35, 94, 55, 23, 81, 39, 6, 49, 77], [31, 95, 38, 53, 10, 75, 6, 55, 90, 14], [2, 82, 50, 87, 64, 70, 85, 38, 57, 19, 92, 71, 48, 9], [29, 91, 49, 93, 6, 33, 36, 84], [31, 54, 29, 60, 47, 80, 83, 62, 31, 68, 30, 29, 96, 11]]
max_len choosen : 14
Padded ids: [[31, 53, 29, 73, 69, 0, 0, 0, 0, 0, 0, 0, 0, 0], [10, 45, 42, 28, 48, 72, 13, 0, 0, 0, 0, 0, 0, 0], [97, 2, 98, 99, 38, 37, 23, 33, 63, 39, 78, 54, 0, 0], [61, 30, 88, 19, 33, 50, 89, 6, 65, 35, 74, 6, 0, 0], [10, 35, 94, 55, 23, 81, 39, 6, 49, 77, 0, 0, 0, 0], [31, 95, 38, 53, 10, 75, 6, 55, 90, 14, 0, 0, 0, 0], [2, 82, 50, 87, 64, 70, 85, 38, 57, 19, 92, 71, 48, 9], [29, 91, 49, 93, 6, 33, 36, 84, 0, 0, 0, 0, 0, 0], [31, 54, 29, 60, 47, 80, 83, 62, 31, 68, 30, 29, 96, 11]]


In [ ]:
# Tensors (X define before it is used)
X = torch.tensor(padded_ids , dtype=torch.long)
Y = torch.tensor(labels , dtype=torch.float32).reshape(-1,1)
print(X.shape) # num reviews , max len
print(Y.shape) #num_reviews , 1

torch.Size([9, 14])
torch.Size([9, 1])


In [ ]:
# The brain (Pytorch Module)
class SentimentNet(nn.Module):

    def __init__(self , vocab_size , dim = 8):
        super().__init__()
        self.emb = nn.Embedding(vocab_size , dim) 
        self.fc = nn.Linear(dim , 2)

    def forward(self , ids):
        pooled = self.emb(ids).mean(dim = 0  , keepdim = True)
        return self.fc(pooled)
